In [1]:
# 1_PPO_Math_Core.ipynb
# ==========================================
# 自动批改系统
# 在 ppo_core.py 中写完代码后，运行此 Cell 进行验证
# ==========================================
import torch
import importlib
import tests
import ppo_core

# 每次重新运行前，自动重新加载你修改过的 ppo_core.py
importlib.reload(ppo_core)
importlib.reload(tests)

print(f"PyTorch Version: {torch.__version__} | CUDA: {torch.cuda.is_available()}\n")

tests.check_gae()
tests.check_ppo_loss()

PyTorch Version: 2.10.0+cu128 | CUDA: True

--- Testing GAE Implementation ---
✅ [PASSED] GAE Advantages & Returns (Max rel_error: 3.7253e-08)

--- Testing PPO Clipped Loss ---
✅ [PASSED] PPO Clipped Loss (rel_error: 4.0640e-08)


In [2]:
# import torch
# import torch.optim as optim
# import gymnasium as gym
# import mani_skill.envs
# import imageio
# from IPython.display import Video
# import importlib

# # 重新加载你的奖励函数库，方便你随时修改随时跑
# import rewards_lab
# importlib.reload(rewards_lab)
# from ppo_trainer import PPOAgent, train_ppo

# # ==========================================
# # 1. 定义奖励注入包裹器 (Reward Injection Wrapper)
# # ==========================================
# class CustomRewardWrapper(gym.Wrapper):
#     @property
#     def num_envs(self):
#         return self.env.unwrapped.num_envs
    
#     def step(self, action):
#         obs, orig_reward, terminated, truncated, info = self.env.step(action)
        
#         # 直接从 GPU 显存底层提取物理状态！
#         tcp_pos = self.env.unwrapped.agent.tcp.pose.p
#         cube_pos = self.env.unwrapped.cube.pose.p
#         goal_pos = self.env.unwrapped.goal_site.pose.p
        
#         # 接触状态估算
#         is_grasped = info.get("is_grasped", torch.zeros_like(terminated, dtype=torch.bool))
#         is_touching = torch.norm(tcp_pos - cube_pos, dim=1) < 0.025
        
#         # 🟢 接入你的试验田！
#         custom_reward, meta = rewards_lab.get_total_reward(tcp_pos, cube_pos, goal_pos, is_grasped, is_touching)
#         # print(meta)
#         return obs, custom_reward, terminated, truncated, info

# # ==========================================
# # 2. 实例化 ManiSkill3 环境并开始训练
# # ==========================================
# # 修复：移除 device 参数。ManiSkill3 只要 num_envs > 1 就会自动拉起 GPU 后端
# env = gym.make("PickCube-v1", num_envs=256, obs_mode="state", control_mode="pd_ee_delta_pose")
# env = CustomRewardWrapper(env)
# device = env.unwrapped.device # 让 Agent 直接对齐环境自动分配的设备 (通常是 cuda:0)

# obs_dim = env.observation_space.shape[1]
# act_dim = env.action_space.shape[1]

# agent = PPOAgent(obs_dim, act_dim).to(device)
# optimizer = optim.Adam(agent.parameters(), lr=3e-4, eps=1e-5)

# print(f"🚀 开始 Phase 1 训练 (Reaching Only) on {device}...")
# train_ppo(env, agent, optimizer, num_updates=500, num_steps=50)

# # ==========================================
# # 3. 渲染单局评估录像并展示
# # ==========================================
# print("\n🎬 训练完成，正在渲染验证录像...")
# # 同样移除 eval_env 的 device 参数
# eval_env = gym.make("PickCube-v1", num_envs=1, obs_mode="state", control_mode="pd_ee_delta_pose", render_mode="rgb_array")
# obs, _ = eval_env.reset()
# eval_device = eval_env.unwrapped.device # num_envs=1 时，可能被默认分配在 CPU

# frames = []
# agent.eval()
# for _ in range(150): 
#     with torch.no_grad():
#         # 保证 obs 和 agent 在同一个设备 (GPU)
#         obs_tensor = obs.to(device)
#         action, _, _, _ = agent.get_action_and_value(obs_tensor)
    
#     # 保证 action 和 eval_env 在同一个设备 (防止 eval_env 在 CPU 时发生张量碰撞)
#     action_for_env = action.to(eval_device)
#     obs, _, _, _, _ = eval_env.step(action_for_env)
    
#     frame = eval_env.render()
#     if isinstance(frame, list): frame = frame[0] 
#     frames.append(frame.squeeze().cpu().numpy())

# video_path = "phase1_reaching.mp4"
# imageio.mimsave(video_path, frames, fps=30)
# eval_env.close()
# env.close()

# print(f"✅ 录像已保存至 {video_path}！")

In [3]:
# 加载 TensorBoard 扩展并在 notebook 内嵌显示
%load_ext tensorboard
%tensorboard --logdir runs

Launching TensorBoard...

In [ ]:
import torch
import torch.optim as optim
import gymnasium as gym
import mani_skill.envs
import imageio
from IPython.display import Video
import importlib
from torch.utils.tensorboard import SummaryWriter
import sapien
import numpy as np
import os
import rewards_lab
importlib.reload(rewards_lab)
from ppo_trainer import PPOAgent, train_ppo

# ==========================================
# 1. 定义奖励注入包裹器
# ==========================================
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
        
        # 算出每个关节的物理中点 M 和活动半径 R
        qlimits = self.env.unwrapped.agent.robot.get_qlimits()
        
        # 2. 用省略号 '...' 穿透所有批次维，强行切出前7个臂关节：形状稳稳锁定为 [..., 7, 2]
        arm_limits = qlimits[..., :7, :]
        arm_limits = arm_limits.to(device=self.env.unwrapped.device, dtype=torch.float32)
        
        # 3. 沿最后一维剥离出 min 和 max，算完后 q_mid 的形状自动变成 [..., 7]
        self.q_mid = (arm_limits[..., 0] + arm_limits[..., 1]) / 2.0
        self.q_range = (arm_limits[..., 1] - arm_limits[..., 0]) / 2.0

    @property
    def num_envs(self):
        return self.env.unwrapped.num_envs

    def step(self, action):
        obs, orig_reward, terminated, truncated, info = self.env.step(action)
        
        tcp_pos = self.env.unwrapped.agent.tcp.pose.p
        cube_pos = self.env.unwrapped.cube.pose.p
        goal_pos = self.env.unwrapped.goal_site.pose.p
        
        is_grasped = info.get("is_grasped", torch.zeros_like(terminated, dtype=torch.bool))
        is_touching = torch.norm(tcp_pos - cube_pos, dim=1) < 0.01
        
        cube_z = cube_pos[:, 2]
        is_dropped = cube_z < 0.01 
        terminated = terminated | is_dropped

        # -----------------------------------------------------------------
        # ⚡ 核心提速点：直接从底层物理引擎显存里，把前7个关节的实时弧度切片抽出来！
        # -----------------------------------------------------------------
        qpos = self.env.unwrapped.agent.robot.qpos[:, :7]

        # 喂给你的 Reward Lab
        custom_reward, reward_info = rewards_lab.get_total_reward(
            tcp_pos, cube_pos, goal_pos, is_grasped, is_touching, 
            qpos, self.q_mid, self.q_range  # 传入缓存好的静态参数
        )
        info.update(reward_info)
        
        return obs, custom_reward, terminated, truncated, info

def evaluate_and_record(agent, eval_env, device, eval_device, video_path):
    obs, _ = eval_env.reset()
    frames = []
    
    # =====================================================================
    # 🔮 [全息视觉升级] 1. 在 SAPIEN 渲染引擎中注入一个“幽灵目标点”
    # =====================================================================
    # scene = eval_env.unwrapped.scene
    # builder = scene.create_actor_builder()
    
    # # 1. 显式创建 PBR 材质
    # mat = sapien.render.RenderMaterial()
    # mat.base_color = [0.0, 0.5, 1.0, 0.8]  # RGBA 赛博蓝
    # mat.roughness = 0.2                    # 降低粗糙度，让它带有一点点光泽
    # mat.emission = [0.0, 0.2, 0.6, 1.0]    # 注入灵魂：开启微弱的“自发光”，防止被机械臂阴影遮挡！

    # # 2. 通过 material 参数传入
    # builder.add_sphere_visual(radius=0.01, material=mat)
    
    # # 依然保持 Kinematic 铁律
    # marker = builder.build_kinematic(name="reaching_carrot_marker")
    # # =====================================================================

    agent.eval()
    try:
        for _ in range(200): 
            with torch.no_grad():
                obs_tensor = obs.to(device)
                action, _, _, _ = agent.get_action_and_value(obs_tensor)
            
            action_for_env = action.to(eval_device)
            obs, _, _, _, _ = eval_env.step(action_for_env)
            
            # =============================================================
            # 🛰️ [雷达锁敌] 2. 每步计算“动态胡萝卜”的物理坐标并驱动瞬移
            # =============================================================
            # 从 ManiSkill 底层显存抽取出方块的当前坐标 (Shape: [1, 3] 转 numpy)
            # cube_pos = eval_env.unwrapped.cube.pose.p[0].cpu().numpy()
            
            # 根据你 rewards_lab.py 里的设定，阶段一的胡萝卜挂在方块正上方 3cm 处
            # target_pos = cube_pos + np.array([0.0, 0.0, 0.03])
            
            # 驱动全息投影球瞬移到目标点
            # marker.set_pose(sapien.Pose(target_pos))
            # =============================================================
            
            frame = eval_env.render()
            if isinstance(frame, list): frame = frame[0] 
            frames.append(frame.squeeze().cpu().numpy())
            
        imageio.mimsave(video_path, frames, fps=30)
        
    finally:
        # =============================================================
        # 🧹 [安全离场] 3. 录完必须销毁该 Actor，否则下个 Chunk 循环重入会报错
        # =============================================================
        # scene.remove_actor(marker)
        pass
        
    agent.train()

# ==========================================
# 3. 环境初始化与分块训练逻辑
# ==========================================
# 创建保存目录
os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./videos", exist_ok=True)

env = gym.make("PickCube-v1", num_envs=256, obs_mode="state", control_mode="pd_ee_delta_pose")
env = CustomRewardWrapper(env)
device = env.unwrapped.device

# 提前初始化一次评估环境，避免在循环里反复创建和销毁
eval_env = gym.make("PickCube-v1", num_envs=1, obs_mode="state", control_mode="pd_ee_delta_pose", render_mode="rgb_array")
eval_device = eval_env.unwrapped.device

obs_dim = env.observation_space.shape[1]
act_dim = env.action_space.shape[1]

agent = PPOAgent(obs_dim, act_dim).to(device)
optimizer = optim.Adam(agent.parameters(), lr=3e-4, eps=1e-5)
writer = SummaryWriter(log_dir="./runs/pick_cube_experiment_2")

# --- 核心修改：分块训练循环 ---
TOTAL_UPDATES = 2000  # 你想要跑的总轮数
EVAL_FREQ = 200       # 每隔多少轮评估/保存一次

num_chunks = TOTAL_UPDATES // EVAL_FREQ

print(f"🚀 开始终局训练 on {device}...")
print(f"📦 计划总更新: {TOTAL_UPDATES} 轮，每 {EVAL_FREQ} 轮保存一次")

for chunk in range(num_chunks):
    update_offset = chunk * EVAL_FREQ
    print(f"\n==============================================")
    print(f"🔥 开始阶段 {chunk + 1}/{num_chunks} (Updates {update_offset + 1} -> {update_offset + EVAL_FREQ})")
    print(f"==============================================")
    
    # 1. 训练当前块 (传入 update_offset 保证 TensorBoard 接力绘图)
    train_ppo(
        env, agent, optimizer, 
        num_updates=EVAL_FREQ, 
        num_steps=50, 
        writer=writer, 
        update_offset=update_offset
    )
    
    current_total_updates = update_offset + EVAL_FREQ
    
    # 2. 保存神经网络权重
    weight_path = f"./checkpoints/ppo_agent_update_{current_total_updates}.pth"
    torch.save(agent.state_dict(), weight_path)
    print(f"💾 权重已保存至: {weight_path}")
    
    # 3. 渲染评估录像
    video_path = f"./videos/eval_video_update_{current_total_updates}.mp4"
    print(f"🎬 正在录制该阶段评估录像...")
    evaluate_and_record(agent, eval_env, device, eval_device, video_path)
    print(f"✅ 阶段录像已保存: {video_path}")

writer.close()
eval_env.close()
env.close()

print("\n🎉 全部训练结束！你可以去 ./videos 文件夹查看智能体进化的过程了！")



[2026-06-22 05:57:32.039] [SAPIEN] [warning] A PhysX CPU system is being created while PhysX GPU is enabled. You can safely ignore this message if it is intended. To use GPU PhysX, create a sapien.physx.PhysxGpuSystem explicitly and pass it to sapien.Scene constructor.


🚀 开始终局训练 on cuda...
📦 计划总更新: 2000 轮，每 200 轮保存一次

🔥 开始阶段 1/10 (Updates 1 -> 200)
Update 010/200 | R_Total: 0.4702 | R_Track: 0.1615 | R_grasp: 0.0000 | Ent: 6.4925
Update 020/200 | R_Total: 0.3969 | R_Track: 0.1399 | R_grasp: 0.0000 | Ent: 6.5289
Update 030/200 | R_Total: 0.2762 | R_Track: 0.1002 | R_grasp: 0.0000 | Ent: 6.5622
Update 040/200 | R_Total: 0.2273 | R_Track: 0.0841 | R_grasp: 0.0000 | Ent: 6.6008
Update 050/200 | R_Total: 0.1599 | R_Track: 0.0622 | R_grasp: 0.0000 | Ent: 6.6351
Update 060/200 | R_Total: 0.1941 | R_Track: 0.0734 | R_grasp: 0.0000 | Ent: 6.6743
Update 070/200 | R_Total: 0.2019 | R_Track: 0.0761 | R_grasp: 0.0000 | Ent: 6.7153
Update 080/200 | R_Total: 0.1640 | R_Track: 0.0630 | R_grasp: 0.0000 | Ent: 6.7568
Update 090/200 | R_Total: 0.1558 | R_Track: 0.0603 | R_grasp: 0.0000 | Ent: 6.7898
Update 100/200 | R_Total: 0.1637 | R_Track: 0.0630 | R_grasp: 0.0000 | Ent: 6.7979
Update 110/200 | R_Total: 0.1948 | R_Track: 0.0730 | R_grasp: 0.0000 | Ent: 6.8035
Update 